# Forward rendering

jaxCAD's image renderer favors predictable forward rendering: early-exit sphere tracing, reconstructed silhouettes, finite-difference normals, GGX materials, soft shadows, reflections, and refraction. Geometry and constraints remain JAX-native and differentiable; rendered pixels are deliberately not an optimization interface.

In [ ]:
import matplotlib.pyplot as plt

from jaxcad.render import Camera, Material, RenderSettings, Scene, make_gradient_sky, render_scene
from jaxcad.sdf.boolean import Union
from jaxcad.sdf.primitives import Plane, Sphere, Torus
from jaxcad.sdf.transforms import Rotate, Translate

In [ ]:
ground_height = -0.92
geometry = Union(
    Translate(
        Sphere(0.82, material=Material(color=[0.78, 0.06, 0.025], roughness=0.42)),
        [-1.45, -0.098, 0.0],
    ),
    Translate(
        Rotate(
            Torus(
                0.62,
                0.23,
                material=Material(
                    color=[1.0, 0.58, 0.08], roughness=0.28, metallic=0.8, reflectivity=0.35
                ),
            ),
            axis="y",
            angle=0.45,
        ),
        [0.0, -0.068, 0.0],
    ),
    Translate(
        Sphere(0.78, material=Material(color=[0.035, 0.18, 0.72], roughness=0.28)),
        [1.4, -0.138, 0.0],
    ),
    Plane(ground_height, material=Material(color=[0.1, 0.12, 0.16], roughness=0.72)),
    smoothness=0.0,
)
scene = Scene(
    geometry,
    camera=Camera(position=(4.4, 3.5, 7.5), target=(0.0, -0.35, 0.0), fov=0.38),
    light_directions=((0.35, 1.0, 0.55), (-0.65, 0.45, -0.2), (0.15, 0.2, 1.0)),
    light_colors=((1.5, 1.35, 1.15), (0.22, 0.3, 0.5), (0.2, 0.23, 0.3)),
    environment_map=make_gradient_sky(
        sky_color=(0.7, 0.76, 0.88),
        horizon_color=(0.18, 0.27, 0.44),
        ground_color=(0.16, 0.18, 0.24),
    ),
)

In [ ]:
settings = RenderSettings.balanced((240, 320))
image = render_scene(scene, settings)
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.axis("off");

## Quality presets

The named presets make the performance/fidelity trade-off explicit. They are immutable dataclasses, so use `dataclasses.replace` for scene-specific changes.

In [ ]:
presets = [
    RenderSettings.draft((180, 240)),
    RenderSettings.balanced((180, 240)),
    RenderSettings.high_quality((180, 240)),
]
images = [render_scene(scene, preset) for preset in presets]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, label, rendered in zip(axes, ["Draft", "Balanced", "High quality"], images):
    axis.imshow(rendered)
    axis.set_title(label)
    axis.axis("off")
plt.tight_layout()